# AI Level Director Studio

A designer governed workflow for 2D platformer level iteration. This notebook is the reproducible walkthrough of the integrated system. It calls the same `LevelDirectorService` the Gradio app uses, with the **real** prior project adapters: Project 5 generation, Project 6 triage, and Project 3 feedback classification.

**Student:** Robert Mayfield

## Notebook Purpose and System Summary

AI Level Director Studio coordinates three prior capstone projects through adapters and adds candidate lifecycle management, persistence, reporting, and a UI. The designer drives the loop: generate or upload candidates, triage them, send ready candidates to playtest, classify the returned feedback, and decide whether to complete or revise. The system is advisory; it never edits a level or claims to predict fun.

This notebook makes live OpenAI triage calls and runs the local generator and classifier, so re-executing it needs `OPENAI_API_KEY` in `.env`. The saved cell outputs below can be read without a key. Reviewers without a key can still exercise the full workflow offline by running `python app.py` and switching the engine toggle to **Mock**, or by running the test suite with `pytest`.

## Setup and Imports

In [1]:
import sys
from pathlib import Path

def _repo_root() -> Path:
    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / 'src' / 'ai_level_director').exists():
            return cand
    raise RuntimeError('Run this from inside the repository.')

ROOT = _repo_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import random
import pandas as pd
from IPython.display import Markdown, display

from ai_level_director.config import load_environment
from ai_level_director.adapters.project3_feedback import Project3FeedbackAdapter
from ai_level_director.adapters.project5_generator import Project5GeneratorAdapter
from ai_level_director.adapters.project6_triage import Project6TriageAdapter
from ai_level_director.workflow.service import LevelDirectorService
from ai_level_director.ui.view_models import candidate_board_view
from ai_level_director.rendering.ascii_renderer import render_level_ascii

SEED = 42
random.seed(SEED)

load_environment()  # local .env wins over any stale system credentials

OUTPUT_ROOT = ROOT / 'outputs'
SAMPLES = ROOT / 'data' / 'sample_levels'

service = LevelDirectorService(
    OUTPUT_ROOT,
    triage_adapter=Project6TriageAdapter(),
    feedback_adapter=Project3FeedbackAdapter(),
    generator_adapter=Project5GeneratorAdapter(),
)

def board(session):
    """Return the candidate board for a session."""
    return candidate_board_view(session)

def show_triage(candidate):
    """Print the triage result for a candidate."""
    tr = candidate.triage_result
    if not tr:
        print('Not triaged.')
        return
    print('action     :', tr.action)
    print('readiness  :', tr.readiness)
    print('rationale  :', tr.rationale)
    if tr.warnings:
        print('warnings   :', tr.warnings)
    if tr.revision_recommendations:
        print('revisions  :', tr.revision_recommendations)
    if tr.playtest_questions:
        print('playtest Qs:', tr.playtest_questions)

print('Service ready with real adapters (Project 5, 6, 3).')

C:\Users\bjm20\Documents\Udacity AI MBA\Capstone\industry-integrated-level-director-studio\integrations\project6_triage\src\agents.py:56: PydanticAIDeprecationWarning: `Agent(output_retries=...)` is deprecated and will be removed in v2.0. Use `retries={'output': ...}` (or `retries=<int>` to set the same budget for both tool and output retries) instead.
  intent_interpreter = Agent(
C:\Users\bjm20\Documents\Udacity AI MBA\Capstone\industry-integrated-level-director-studio\integrations\project6_triage\src\agents.py:96: PydanticAIDeprecationWarning: `Agent(output_retries=...)` is deprecated and will be removed in v2.0. Use `retries={'output': ...}` (or `retries=<int>` to set the same budget for both tool and output retries) instead.
  triage_director = Agent(
C:\Users\bjm20\Documents\Udacity AI MBA\Capstone\industry-integrated-level-director-studio\integrations\project6_triage\src\agents.py:203: PydanticAIDeprecationWarning: `Agent(output_retries=...)` is deprecated and will be removed in

Service ready with real adapters (Project 5, 6, 3).


## Component Checks

Confirm the three adapters loaded. No API call is made here; triage runs in the scenarios below.

In [2]:
print('Triage adapter   :', type(service.triage_adapter).__name__)
print('Feedback adapter :', type(service.feedback_adapter).__name__)
print('Generator adapter:', type(service.generator_adapter).__name__)
print('Reference levels  :', len(service.triage_adapter._reference_levels))
print('Feedback sanity  :', service.feedback_adapter.classify('fun and fair').sentiment)

Triage adapter   : Project6TriageAdapter
Feedback adapter : Project3FeedbackAdapter
Generator adapter: Project5GeneratorAdapter
Reference levels  : 10
Feedback sanity  : positive


## Scenario A: Generated Candidate Path (Project 5 + Project 6)

Generate a candidate with Project 5, then triage it with Project 6. Generated levels are drafts with derivative risk, so they always flow through triage before any playtest.

In [3]:
service.start_session(
    'An easy, beginner friendly opening segment with a gentle first jump.',
    target_difficulty='easy', session_id='demo-generated',
)
service.add_generated_candidate('demo-generated', n=1, temperature=1.2, seed=SEED)
session_a = service.run_triage('demo-generated', 'G-001')
print(render_level_ascii(session_a.candidates[0].level_text))
show_triage(session_a.candidates[0])
board(session_a)

--------------------------------
--------------------------------
--------------------------------
--------------------------------
--------------------------------
---------SSS----SQQS------------
--------------------------------
--------------------------------
--------------------------------
Q-----S----------SS------E------
-------------------------<>-----
----------------------------<>--
------------------[]--[]-[]-[]--
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
action     : reject_structural
readiness  : not_ready
rationale  : The candidate level contains a structural error: a pipe top is present without a supporting body or ground, which is a fatal integrity issue.
warnings   : ['pipe top without a body or ground support']


,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,G-001,Generated candidate,generated,structural_rejected,reject_structural,not_ready,-,pipe top without a body or ground support,Rejected for structural reasons


**Interpretation.** Triage returned `reject_structural`: the generated candidate had a pipe top with no supporting body, a fatal integrity error, so it was blocked before playtest. This is the safeguard working as intended, not a demo failure. Project 5 produces drafts and Project 6 catches structurally invalid ones before they reach a playtester. A structurally valid generated candidate would instead be accepted, sent for revision, or, when it is too close to a known level, flagged as a derivative draft, shown next.

## Scenario A2: Derivative Draft Disclosure (originality safeguard)

Generated or uploaded candidates can land too close to existing levels. To exercise the originality safeguard directly, we submit a candidate that duplicates a level from Project 6's reference library. The novelty tool reports maximal similarity, so triage should refuse to accept the candidate and route it back rather than send it to playtest. Setting the session novelty preference to `original` tells the agent to weigh originality heavily.

In [4]:
reference_level = (ROOT / 'integrations' / 'project6_triage' / 'reference_levels' / 'ref_01.txt').read_text(encoding='utf-8')
service.start_session(
    'An original easy opening segment. Avoid reusing existing layouts.',
    target_difficulty='easy', novelty_preference='original', session_id='demo-derivative',
)
service.add_uploaded_candidate('demo-derivative', reference_level, title='Near duplicate of a known level')
session_a2 = service.run_triage('demo-derivative', 'U-001')
show_triage(session_a2.candidates[0])
board(session_a2)

action     : flag_as_derivative_draft
readiness  : revise_before_playtest
rationale  : The candidate is structurally valid, easy, and opens with a safe zone appropriate for beginners. However, it has extremely high similarity (1.0) to a known level, which conflicts with the hard constraint to avoid reusing existing layouts and the explicit preference for originality and high novelty.


,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,U-001,Near duplicate of a known level,uploaded,derivative_review_needed,flag_as_derivative_draft,revise_before_playtest,-,-,Human originality review


**Interpretation.** The candidate duplicates a reference level, so the novelty tool reports a similarity of 1.0, the top of its range. Triage detected this directly, its rationale calls the candidate highly derivative, and returned `recommend_revision`, moving it to `revision_needed` with a note to redesign the layout for originality. The originality safeguard fired: a derivative candidate was caught and sent back rather than accepted. The agent can also route such a candidate to `derivative_review_needed` through `flag_as_derivative_draft` for a dedicated human originality decision; because triage is a live language model judgment, the exact action varies between runs, but high similarity to a known level is precisely what this safeguard targets. The studio discloses derivative risk rather than hiding it.

## Scenario B: Uploaded Candidate Path

A designer authored candidate (here a bundled sample) is triaged the same way.

In [5]:
easy_level = (SAMPLES / 'easy_opener.txt').read_text(encoding='utf-8')
service.start_session(
    'A fair, beginner friendly segment. Preserve the layout.',
    target_difficulty='easy', session_id='demo-uploaded',
)
service.add_uploaded_candidate('demo-uploaded', easy_level, title='Easy opener')
session_b = service.run_triage('demo-uploaded', 'U-001')
show_triage(session_b.candidates[0])
board(session_b)

action     : accept_for_playtest
readiness  : ready_for_playtest
rationale  : The candidate is structurally valid, matches the easy difficulty target, and is appropriate for beginners. It opens with a safe zone and contains no hazards or enemies, ensuring a fair and friendly experience. The novelty is medium, which aligns with the brief. Pacing is flat, which is suitable for an easy, beginner-friendly segment.
playtest Qs: ['Does the absence of hazards or enemies make the segment engaging enough for beginners, or does it feel too empty?', 'Do the coin placements provide enough guidance and reward for new players?']


,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,U-001,Easy opener,uploaded,ready_for_playtest,accept_for_playtest,ready_for_playtest,-,-,Send to playtest


## Scenario C: Positive Playtest Feedback (Project 3 completes the loop)

A ready candidate is sent to playtest; positive feedback classified by Project 3 completes it. The send step runs only if triage marked the candidate ready, since live triage may decide otherwise.

In [6]:
service.start_session(
    'A fair beginner segment that should not feel frustrating.',
    target_difficulty='easy', session_id='demo-positive',
)
service.add_uploaded_candidate('demo-positive', easy_level, title='Easy opener')
session_c = service.run_triage('demo-positive', 'U-001')
cand = session_c.candidates[0]
show_triage(cand)
if cand.workflow_state == 'ready_for_playtest':
    service.send_to_playtest('demo-positive', 'U-001')
    session_c = service.submit_feedback('demo-positive', 'U-001', 'Great pacing, the first jump felt fair and fun.')
else:
    print('Triage did not mark the candidate ready; state:', cand.workflow_state)
board(session_c)

action     : accept_for_playtest
readiness  : ready_for_playtest
rationale  : The candidate is structurally valid, matches the easy difficulty target, and provides a safe, frustration-free experience for beginners. There are no hazards or enemies, ensuring fairness and no frustration. The novelty is medium, which aligns with the brief. Pacing is flat, which is appropriate for a beginner segment. No difficulty spikes or structural issues are present.
playtest Qs: ['Does the lack of any hazards or challenges make the segment too uneventful, even for beginners?', 'Do the coin and collectible placements provide enough engagement for new players?']


,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,U-001,Easy opener,uploaded,complete,accept_for_playtest,ready_for_playtest,positive,-,Complete


## Scenario D: Negative Playtest Feedback and Revision

Negative feedback sends the candidate to revision_needed, and the designer creates a revised candidate that points back to the original.

In [7]:
service.start_session(
    'A fair beginner segment. The first jump must not feel unfair.',
    target_difficulty='easy', session_id='demo-negative',
)
service.add_uploaded_candidate('demo-negative', easy_level, title='Easy opener')
session_d = service.run_triage('demo-negative', 'U-001')
cand = session_d.candidates[0]
if cand.workflow_state == 'ready_for_playtest':
    service.send_to_playtest('demo-negative', 'U-001')
    session_d = service.submit_feedback('demo-negative', 'U-001', 'The first jump felt unfair and frustrating.')
    if session_d.candidates[0].workflow_state == 'revision_needed':
        revised = (SAMPLES / 'beginner_excitement.txt').read_text(encoding='utf-8')
        session_d = service.create_revised_candidate('demo-negative', 'U-001', revised, notes='Reworked the first jump.')
else:
    print('Triage did not mark the candidate ready; state:', cand.workflow_state)
board(session_d)

,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,U-001,Easy opener,uploaded,revision_needed,accept_for_playtest,ready_for_playtest,negative,-,Create a revised candidate
1,R-001,Revision of U-001,revised,draft,-,-,-,-,Run triage


## Scenario E: Ambiguous or Conflicting Brief (responsible control)

A self contradictory brief should make Project 6 ask for clarification or escalate to human review rather than force a recommendation.

In [8]:
service.start_session(
    'A relaxing, beginner friendly segment that is also brutally hard with constant enemies and huge gaps.',
    session_id='demo-conflicting',
)
service.add_uploaded_candidate('demo-conflicting', easy_level, title='Easy opener')
session_e = service.run_triage('demo-conflicting', 'U-001')
show_triage(session_e.candidates[0])
board(session_e)

action     : request_clarification
readiness  : not_ready
rationale  : The interpreted intent contains conflicting requirements: it asks for both 'beginner friendly' and 'hard' difficulty, as well as a 'relaxing' feel alongside 'constant enemies and huge gaps.' These are mutually exclusive and make it impossible to confidently judge the candidate against a clear target.


,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,U-001,Easy opener,uploaded,clarification_needed,request_clarification,not_ready,-,-,"Clarify the brief, then revise"


## Scenario F: Short Feedback Reliability and Designer Override

Project 3's analysis found the classifier misreads short reviews, so short playtest feedback is flagged and the designer can override the label after reading the text. Here a short, vague note is classified, flagged as low reliability, and the designer overrides the call. The classifier's original label is kept for the audit trail.

In [9]:
service.start_session(
    'A fair, beginner friendly opening segment.',
    target_difficulty='easy', session_id='demo-override',
)
service.add_uploaded_candidate('demo-override', easy_level, title='Easy opener')
session_f = service.run_triage('demo-override', 'U-001')
cand = session_f.candidates[0]
if cand.workflow_state == 'ready_for_playtest':
    service.send_to_playtest('demo-override', 'U-001')
    session_f = service.submit_feedback('demo-override', 'U-001', 'Looks pretty fun.')
    rec = session_f.candidates[0].feedback_records[-1]
    print('classifier label     :', rec.feedback_result.sentiment)
    print('reliability flag     :', rec.warning)
    print('state after feedback :', session_f.candidates[0].workflow_state)
    # The designer reads the short note and overrides the classifier's call.
    session_f = service.override_feedback('demo-override', 'U-001', 'negative')
    rec = session_f.candidates[0].feedback_records[-1]
    print('after override       :', session_f.candidates[0].workflow_state)
    print('classifier kept      :', rec.feedback_result.sentiment, '| designer override:', rec.designer_override)
else:
    print('Triage did not mark the candidate ready; state:', cand.workflow_state)
board(session_f)

classifier label     : positive
reliability flag     : Short feedback (3 words). The classifier is less reliable on short reviews and can misread short negative feedback as positive, so review the text before trusting this label.
state after feedback : complete
after override       : revision_needed
classifier kept      : positive | designer override: negative


,candidate_id,title,source,state,triage_action,readiness,feedback,main_warning,next_step
0,U-001,Easy opener,uploaded,revision_needed,accept_for_playtest,ready_for_playtest,positive,-,Create a revised candidate


**Interpretation.** The feedback was short, so the system attached a reliability warning regardless of the label, prompting the designer to read the text. The designer then overrode the classifier's call: the candidate moved to `revision_needed`, the classifier's original label stayed on the record, and the override was logged. This makes the classifier's advisory nature concrete , the model proposes, the designer disposes.

## Results Summary and Output Reports

Generate a Markdown session report for one scenario. Reports are pure formatting over saved state and always disclose the system's limitations.

In [10]:
report_path = service.build_session_report('demo-generated')
print('Report written to:', report_path)
display(Markdown(report_path.read_text(encoding='utf-8')))

Report written to: c:\Users\bjm20\Documents\Udacity AI MBA\Capstone\industry-integrated-level-director-studio\outputs\reports\demo-generated_report.md


# AI Level Director Studio Session Report

## Session

- Session ID: `demo-generated`
- Status: active
- Design brief: An easy, beginner friendly opening segment with a gentle first jump.
- Target difficulty: easy
- Novelty preference: unspecified
- Candidates: 1
- Created: 2026-06-25T21:32:57.586052+00:00
- Updated: 2026-06-25T21:33:15.399404+00:00

## Candidate Summary

| ID | Title | Source | State | Triage | Readiness | Feedback | Warning |
|---|---|---|---|---|---|---|---|
| G-001 | Generated candidate | generated | structural_rejected | reject_structural | not_ready | - | pipe top without a body or ground support |

## Candidates by State

- structural_rejected: 1

## Candidate Details



### G-001: Generated candidate (generated)

- State: structural_rejected
- Iteration: 1
- Generation: {'source_project': 'Project 5', 'target_difficulty': 'easy', 'temperature': 1.2, 'seed': 42}

**Triage:** reject_structural (readiness: not_ready)

The candidate level contains a structural error: a pipe top is present without a supporting body or ground, which is a fatal integrity issue.

Warnings:
- pipe top without a body or ground support

History:
- 2026-06-25T21:33:03.765403+00:00 generated: Candidate G-001 created from generated source.
- 2026-06-25T21:33:15.399404+00:00 triaged: Triaged: reject_structural -> structural_rejected.

## Limitations and Responsible Use

- Generated candidates are drafts, not final assets. They can be derivative or too similar to the generator's training corpus, so novelty and derivative risk are flagged rather than hidden.
- Playtest feedback classification is a broad reception signal, not a complete playtest analysis. It returns a label only, with no calibrated confidence, and does not explain exact design causes.
- Heuristic difficulty is a structural estimate, not player validated difficulty.
- The system is advisory and designer governed. It never edits or finalizes a level on its own, and it does not claim to predict whether a level is fun.


## Safeguards, Limitations, and Final Notes

AI Level Director Studio is a designer governed compound AI workflow: Project 5 supplies candidate drafts, Project 6 is the single triage authority, and Project 3 turns playtest feedback into a reception signal, while Project 7 owns orchestration, candidate lifecycle state, persistence, and reporting. Across the scenarios the system interprets a brief, triages each candidate, routes ready candidates through playtest, and records every decision in per candidate state and an append only event log. The main integration challenge was the seams between three independently built systems, kept decoupled behind adapters and used as built rather than papering over their limits. Those limits are surfaced, not hidden: generated levels carry derivative risk; the feedback classifier is trained on game reviews and misreads short notes, so short feedback is flagged and the designer can override its call; difficulty is heuristic; and the system is advisory, so the designer makes every final decision.